[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sterolose/sirius-statistics-2026/blob/main/notebooks/4_two_means.ipynb)

## Семинар 4. Сравнение средних двух выборок

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

Поработаем с датасетом [Anorexia](https://vincentarelbundock.github.io/Rdatasets/doc/MASS/anorexia.html). Он содержит измерения веса 72 молодых пациенток с нервной анорексией до и после периода наблюдения.

Столбцы содержат следующие переменные:
- `rowname` — индекс пациентки;
- `Treat` — группа исследования:
  - `Cont` — контрольная группа, 26 пациенток;
  - `CBT` — когнитивно-поведенческая терапия, 29 пациенток;
  - `FT` — семейная терапия, 17 пациенток.
- `Prewt` — вес до периода наблюдения, в фунтах;
- `Postwt` — вес после периода наблюдения, в фунтах.

Скачаем CSV-файл в папку `../data`. Команда `mkdir -p` создаст ее, если она отсутствует.

In [ ]:
%%bash
mkdir -p ../data
wget -O ../data/anorexia.csv https://vincentarelbundock.github.io/Rdatasets/csv/MASS/anorexia.csv

In [ ]:
df = pd.read_csv("../data/anorexia.csv")
df.head(2)

Давайте сделаем датафрейм чуть читабельнее и переведем вес в килограммы.

In [ ]:
df = df.drop(columns="rownames")
df = df.rename(
    columns={
        "Treat": "treatment",
        "Prewt": "W_before",
        "Postwt": "W_after",
    }
)

df["W_before"] = df["W_before"] * 0.45359237
df["W_after"] = df["W_after"] * 0.45359237

df.head(2)

Как обычно, во всех заданиях используем уровень значимости $\alpha = 0.05$ и двусторонние альтернативы.

### Задание 1

Оценим нормальность распределения для всего датасета. Построим гистограмму и Q–Q plot как для исходного, так и для конечной массы тела, а затем выполним тест Шапиро — Уилка.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

w_cols = ["W_before", "W_after"]
w_values = df[w_cols].to_numpy()
bins = np.arange(w_values.min(), w_values.max() + 1)

for var in w_cols:
    values = df[var]
    axes[0].hist(values, bins=bins, alpha=0.7, label=var)

    z = stats.zscore(values, ddof=1)
    theoretical, observed = stats.probplot(z, dist="norm", fit=False)
    axes[1].scatter(theoretical, observed, s=18, alpha=0.7)

axes[1].plot([-3, 3], [-3, 3], "k--", linewidth=1, zorder=0)
axes[1].set(xlim=(-3, 3), ylim=(-3, 3))
axes[0].legend()

plt.tight_layout()
plt.show()

In [ ]:
pd.Series({
    var: stats.shapiro(df[var]).pvalue for var in ["W_before", "W_after"]
}, name="p-value").to_frame()

Повторите аналогичную проверку нормальности обоих переменных для всех трех групп.  
Графики изобразите на сетке 3 × 2, а результаты тестов оформите через `pd.DataFrame`.

In [ ]:
# Your code here

С помощью `.groupby` оцените среднее и стандартное отклонение отдельно для трех групп.

In [ ]:
# Your code here

### Задание 2

Сравним исходный вес в группах `CBT` и `Cont` с помощью t-критерия. За что отвечает параметр `equal_var`? Попробуем выполнить тест с разными его значениями.

In [ ]:
control = df.loc[df["treatment"] == "Cont", "W_before"]
cbt = df.loc[df["treatment"] == "CBT", "W_before"]

student = stats.ttest_ind(control, cbt, equal_var=True)
welch = stats.ttest_ind(control, cbt, equal_var=False)

pd.DataFrame({
    "test": ["Student", "Welch"],
    "statistic": [student.statistic, welch.statistic],
    "p-value": [student.pvalue, welch.pvalue],
})

Далее по умолчанию будем использовать тест Уэлча. Аналогично выполните проверку на различие среднего веса после терапии для этих двух групп.  
Отвергаем ли мы нулевую гипотезу?

In [ ]:
# Your code here

### Задание 3

Для оценки динамики веса у одной и той же пациентки введем новую переменную:
$$
\Delta W = W_{\mathrm{after}} - W_{\mathrm{before}}.
$$

In [ ]:
df["delta_W"] = df["W_after"] - df["W_before"]
df.groupby("treatment", sort=False)["delta_W"].agg(["mean", "std"]).T.round(2)

Оцените распределение новой величины на нормальность для группы `FT`.

In [ ]:
# Your code here

### Задание 4

С помощью парного теста Стьюдента роверьте, изменился ли вес в группе `FT` в среднем. Для этого используйте `stats.ttest_rel`.

In [ ]:
# Your code here

Проверьте нулевую гипотезу о том, что среднее изменение веса равно нулю, с помощью одновыборочного t-теста для `delta_W`. Для этого используйте `stats.ttest_1samp` с параметром `popmean=0`. Сравните статистику и p-value с результатом парного t-теста.

In [ ]:
# Your code here

### Задание 5

Проверьте, различается ли среднее изменение веса между группами `CBT` и `FT` с помощью двухвыборочного t-теста Уэлча. Выведите значение статистики и p-value.

In [ ]:
# Your code here